# GIAI ĐOẠN 4: KỸ THUẬT ĐẶC TRƯNG NGHIỆP VỤ & PHÂN CHIA DỮ LIỆU (FEATURE ENGINEERING & 3-WAY SPLIT)
## Data Preparation: Thiết kế Đặc trưng Phi tuyến tính & Khóa Bộ Dữ liệu Kiểm thử Độc lập
### Doanh nghiệp giả định: EuroVanguard Bank (France, Germany, Spain)
**Quy trình chuẩn:** CRISP-DM Phase 4 — Data Preparation

---
### Mục tiêu giai đoạn:
1. **Thiết kế 6 Đặc trưng Nghiệp vụ (Domain Features):** Giải quyết các quy luật phi tuyến tính đã phát hiện từ EDA (Phân bố 2 đỉnh ở Balance, Đường cong chữ U ngược ở NumOfProducts, Bẫy bán chéo theo độ tuổi, và Tương tác đa chiều).
2. **Ngăn chặn Tuyệt đối Rò rỉ Dữ liệu (Strict Leakage Prevention):** Các đặc trưng được tạo theo cơ chế xác định từng dòng độc lập (Row-wise deterministic), không phụ thuộc vào trạng thái toàn cục của tập dữ liệu.
3. **Phân chia 3 Tập Dữ liệu Chuẩn mực (3-Way Stratified Split):**
   - **Tập Huấn luyện (Train Set - 70% = 7,000 dòng):** Dùng để huấn luyện mô hình và chạy Stratified 5-Fold Cross Validation.
   - **Tập Thẩm định (Validation Set - 15% = 1,500 dòng):** Dùng để so sánh mô hình, tinh chỉnh siêu tham số và tối ưu ngưỡng phân loại $T^*$.
   - **Tập Kiểm thử (Test Set - 15% = 1,500 dòng):** Khóa kín tuyệt đối (Untouched / Out-of-Sample Holdout), chỉ mở duy nhất 1 lần để đánh giá hiệu năng tổng thể kèm khoảng tin cậy 95% Bootstrap CI.


## 1. Thiết lập Môi trường & Nạp Module Chuẩn hóa

In [1]:
import os
import sys

# Xác định thư mục gốc dự án (chạy an toàn tuyệt đối ở cả VS Code workspace root và Jupyter notebooks/)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'src')):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
%matplotlib inline

# Nạp module nội bộ của dự án
from src.preprocessing.feature_engineering import (
    create_domain_features,
    split_dataset_3way,
    save_processed_splits
)
from src.eda.statistical_tests import calculate_woe_iv

INTERIM_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'interim', 'churn_technical_cleaned.csv')
df_clean = pd.read_csv(INTERIM_DATA_PATH)
print(f"Đã nạp dữ liệu sạch: {df_clean.shape[0]:,} dòng x {df_clean.shape[1]} cột")


Đã nạp dữ liệu sạch: 10,000 dòng x 19 cột


## 2. Tạo 6 Đặc trưng Nghiệp vụ (Domain-Engineered Features)

Dựa trên các bằng chứng thống kê từ Giai đoạn 3 (EDA & Diagnostic Analytics), 6 đặc trưng sau được tạo lập:
1. **`IsZeroBalance`:** Cờ nhị phân (1 nếu Balance == 0.0, ngược lại 0). Giải quyết bẫy phân bố 2 đỉnh.
2. **`Balance_To_Salary_Ratio`:** Tỷ lệ số dư tiền gửi trên mức lương ước tính (`Balance / (Salary + 1)`).
3. **`Products_Per_Tenure`:** Tốc độ mua thêm sản phẩm theo thời gian gắn bó (`NumOfProducts / (Tenure + 1)`).
4. **`Product_Holding_Risk_Tier`:** Phân tầng rủi ro bán chéo (`Optimal` cho 2 sp, `Moderate_Risk` cho 1 sp, `Extreme_Danger` cho 3-4 sp).
5. **`Age_Lifecycle_Stage`:** Phân đoạn giai đoạn vòng đời (`<30`, `30-40`, `41-50`, `51-60`, `>60`).
6. **`Engagement_Score`:** Điểm tương tác tổng hợp tích hợp từ trạng thái hoạt động, thẻ tín dụng và số lượng sản phẩm.


In [2]:
df_engineered = create_domain_features(df_clean)
print(f"Kích thước sau khi tạo đặc trưng: {df_engineered.shape[0]:,} dòng x {df_engineered.shape[1]} cột")

# Hiển thị 5 dòng mẫu của các đặc trưng mới tạo
engineered_cols = [
    'IsZeroBalance', 'Balance_To_Salary_Ratio', 'Products_Per_Tenure',
    'Product_Holding_Risk_Tier', 'Age_Lifecycle_Stage', 'Engagement_Score'
]
display(df_engineered[['CustomerId', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'Exited'] + engineered_cols].head())


Kích thước sau khi tạo đặc trưng: 10,000 dòng x 25 cột


,CustomerId,Age,Tenure,Balance,NumOfProducts,Exited,IsZeroBalance,Balance_To_Salary_Ratio,Products_Per_Tenure,Product_Holding_Risk_Tier,Age_Lifecycle_Stage,Engagement_Score
0,15634602,42,2,0.00,1,1,1,0.0000,0.3333,Moderate_Risk,41-50,3
1,15647311,41,1,83807.86,1,0,0,0.7447,0.5000,Moderate_Risk,41-50,2
2,15619304,42,8,159660.80,3,1,0,1.4014,0.3333,Extreme_Danger,41-50,1
3,15701354,39,1,0.00,2,0,1,0.0000,1.0000,Optimal,30-40,1
4,15737888,43,2,125510.82,1,0,0,1.5870,0.3333,Moderate_Risk,41-50,3


## 3. Thẩm định Thống kê & Sức mạnh Dự báo của Các Đặc trưng Mới

In [3]:
# Tính toán Churn Rate và IV cho các đặc trưng mới
print("=== THỐNG KÊ RỜI BỎ THEO CÁC ĐẶC TRƯNG MỚI ===")

for col in ['Product_Holding_Risk_Tier', 'Age_Lifecycle_Stage', 'Engagement_Score', 'IsZeroBalance']:
    summary = df_engineered.groupby(col)['Exited'].agg(
        Total='count',
        Churned='sum',
        Churn_Rate_Pct=lambda x: round(x.mean() * 100, 2)
    ).reset_index()
    
    woe_df, iv_val = calculate_woe_iv(df_engineered, col, 'Exited')
    print(f"\n--- Đặc trưng: {col} (IV = {iv_val:.4f}) ---")
    display(summary)


=== THỐNG KÊ RỜI BỎ THEO CÁC ĐẶC TRƯNG MỚI ===

--- Đặc trưng: Product_Holding_Risk_Tier (IV = 0.9185) ---


,Product_Holding_Risk_Tier,Total,Churned,Churn_Rate_Pct
0,Extreme_Danger,326,280,85.89
1,Moderate_Risk,5084,1409,27.71
2,Optimal,4590,349,7.60



--- Đặc trưng: Age_Lifecycle_Stage (IV = 0.7411) ---


,Age_Lifecycle_Stage,Total,Churned,Churn_Rate_Pct
0,30-40,4451,539,12.11
1,41-50,2320,788,33.97
2,51-60,797,448,56.21
3,<30,1968,148,7.52
4,>60,464,115,24.78



--- Đặc trưng: Engagement_Score (IV = 0.5061) ---


,Engagement_Score,Total,Churned,Churn_Rate_Pct
0,0,768,294,38.28
1,1,2570,862,33.54
2,2,2344,354,15.10
3,3,2583,438,16.96
4,4,1735,90,5.19



--- Đặc trưng: IsZeroBalance (IV = 0.0996) ---


,IsZeroBalance,Total,Churned,Churn_Rate_Pct
0,0,6383,1538,24.10
1,1,3617,500,13.82


## 4. Phân chia Dữ liệu 3 Tập Phân tầng (70% Train / 15% Val / 15% Test)

In [4]:
train_df, val_df, test_df = split_dataset_3way(
    df_engineered,
    target_col='Exited',
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=42
)

print(f"Tập Huấn luyện (Train): {train_df.shape[0]:,} dòng ({train_df.shape[0]/len(df_engineered)*100:.1f}%)")
print(f"Tập Thẩm định (Validation): {val_df.shape[0]:,} dòng ({val_df.shape[0]/len(df_engineered)*100:.1f}%)")
print(f"Tập Kiểm thử (Test): {test_df.shape[0]:,} dòng ({test_df.shape[0]/len(df_engineered)*100:.1f}%)")


Tập Huấn luyện (Train): 7,000 dòng (70.0%)


Tập Thẩm định (Validation): 1,500 dòng (15.0%)
Tập Kiểm thử (Test): 1,500 dòng (15.0%)


### 4.1. Kiểm tra Tính Độc lập Tuyệt đối (Leakage Check) & Cân bằng Phân tầng

In [5]:
# 1. Kiểm tra trùng lặp ID khách hàng giữa 3 tập
train_ids = set(train_df['CustomerId'])
val_ids = set(val_df['CustomerId'])
test_ids = set(test_df['CustomerId'])

leakage_train_val = len(train_ids.intersection(val_ids))
leakage_train_test = len(train_ids.intersection(test_ids))
leakage_val_test = len(val_ids.intersection(test_ids))

print(f"Số lượng ID trùng giữa Train và Val: {leakage_train_val} (Kỳ vọng: 0)")
print(f"Số lượng ID trùng giữa Train và Test: {leakage_train_test} (Kỳ vọng: 0)")
print(f"Số lượng ID trùng giữa Val và Test: {leakage_val_test} (Kỳ vọng: 0)")
assert leakage_train_val == 0 and leakage_train_test == 0 and leakage_val_test == 0, "LỖI RÒ RỈ DỮ LIỆU!"

# 2. So sánh phân bố tỷ lệ Churn và các biến cốt lõi giữa 3 tập
split_comparison = pd.DataFrame({
    'Metric': [
        'Quy mô (Rows)',
        'Tỷ lệ Rời bỏ Churn (%)',
        'Tuổi trung bình (Age)',
        'Số dư trung bình (Balance EUR)',
        'Tỷ lệ Khách hàng Đức (%)',
        'Tỷ lệ Khách hàng Nữ (%)',
        'Tỷ lệ Khách hàng Tích cực (%)'
    ],
    'Train (70%)': [
        f"{len(train_df):,}",
        f"{train_df['Exited'].mean()*100:.2f}%",
        f"{train_df['Age'].mean():.2f}",
        f"{train_df['Balance'].mean():,.2f}",
        f"{(train_df['Geography'] == 'Germany').mean()*100:.2f}%",
        f"{(train_df['Gender'] == 'Female').mean()*100:.2f}%",
        f"{train_df['IsActiveMember'].mean()*100:.2f}%"
    ],
    'Validation (15%)': [
        f"{len(val_df):,}",
        f"{val_df['Exited'].mean()*100:.2f}%",
        f"{val_df['Age'].mean():.2f}",
        f"{val_df['Balance'].mean():,.2f}",
        f"{(val_df['Geography'] == 'Germany').mean()*100:.2f}%",
        f"{(val_df['Gender'] == 'Female').mean()*100:.2f}%",
        f"{val_df['IsActiveMember'].mean()*100:.2f}%"
    ],
    'Test (15%)': [
        f"{len(test_df):,}",
        f"{test_df['Exited'].mean()*100:.2f}%",
        f"{test_df['Age'].mean():.2f}",
        f"{test_df['Balance'].mean():,.2f}",
        f"{(test_df['Geography'] == 'Germany').mean()*100:.2f}%",
        f"{(test_df['Gender'] == 'Female').mean()*100:.2f}%",
        f"{test_df['IsActiveMember'].mean()*100:.2f}%"
    ]
})

display(split_comparison)


Số lượng ID trùng giữa Train và Val: 0 (Kỳ vọng: 0)
Số lượng ID trùng giữa Train và Test: 0 (Kỳ vọng: 0)
Số lượng ID trùng giữa Val và Test: 0 (Kỳ vọng: 0)


,Metric,Train (70%),Validation (15%),Test (15%)
0,Quy mô (Rows),"7,000","1,500","1,500"
1,Tỷ lệ Rời bỏ Churn (%),20.39%,20.40%,20.33%
2,Tuổi trung bình (Age),38.95,38.85,38.88
3,Số dư trung bình (Balance EUR),"76,656.24","74,834.01","77,342.78"
4,Tỷ lệ Khách hàng Đức (%),25.16%,24.73%,25.13%
5,Tỷ lệ Khách hàng Nữ (%),44.74%,46.67%,47.40%
6,Tỷ lệ Khách hàng Tích cực (%),51.97%,50.07%,50.80%


## 5. Lưu trữ Các Tập Dữ liệu Đã Tiền Xử lý (Data Serialization)

In [6]:
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
saved_paths = save_processed_splits(train_df, val_df, test_df, output_dir=PROCESSED_DIR)

# Lưu thêm tập full đã feature-engineered để phục vụ tham chiếu
full_path = os.path.join(PROCESSED_DIR, 'full_engineered.csv')
df_engineered.to_csv(full_path, index=False)
saved_paths['full'] = full_path

print("Đã lưu trữ thành công các tập dữ liệu:")
for key, p in saved_paths.items():
    size_mb = os.path.getsize(p) / 1024
    print(f" - [{key.upper()}] {p} ({size_mb:.1f} KB)")


Đã lưu trữ thành công các tập dữ liệu:


 - [TRAIN] D:\Machine_Learning_UDA\Bank_Customer_Churn\data\processed\train.csv (811.5 KB)
 - [VAL] D:\Machine_Learning_UDA\Bank_Customer_Churn\data\processed\val.csv (174.5 KB)
 - [TEST] D:\Machine_Learning_UDA\Bank_Customer_Churn\data\processed\test.csv (174.2 KB)
 - [FULL] D:\Machine_Learning_UDA\Bank_Customer_Churn\data\processed\full_engineered.csv (1159.5 KB)


## 6. Tổng kết Bàn giao Sang Giai đoạn 5 (Predictive Modeling)

1. **Dữ liệu Huấn luyện Đã Sẵn sàng:**
   - 7,000 hồ sơ khách hàng tại `data/processed/train.csv` với đầy đủ 24 trường dữ liệu (18 trường gốc + 6 đặc trưng kỹ thuật).
   - Đảm bảo tỷ lệ churn 20.39% đồng nhất.
2. **Nguyên tắc Quản trị Mô hình:**
   - Tập `test.csv` (1,500 hồ sơ) được niêm phong hoàn toàn, không tham gia vào quá trình huấn luyện hay tinh chỉnh siêu tham số.
   - Tập `val.csv` (1,500 hồ sơ) sẽ được dùng để chọn mô hình quán quân giữa Logistic Regression, Decision Tree, Random Forest, và XGBoost/HistGradientBoosting.
3. **Kịch bản Khử rò rỉ (Ablation Scenarios):**
   - Loại bỏ `Complain` trong kịch bản dự báo sớm (Early-Warning Proactive Model).
   - Đánh giá kịch bản công bằng phi định kiến đạo đức AI (Ablation không dùng `Gender`).
